<!-- CODEx bilingual cell explanation: start -->
### Cell 01 — 环境、路径与碳分析依赖 / Environment, paths, and carbon-analysis dependencies

**中文说明**：导入碳排放计算、模型重建、交叉验证和绘图所需库，建立 Step 4 输出目录，并定义 Step 1-3 的输入文件路径。该 cell 是 EUI-OCEI 耦合分析的运行入口。

**输入与依赖**：依赖本地 Python/Jupyter 环境、项目根目录和后续单元需要共享的配置参数。

**主要输出**：建立路径、随机种子、绘图样式、配置字典或输出目录等基础对象。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Imports the dependencies for carbon accounting, model reconstruction, cross-validation, and plotting; creates Step 4 output directories; and defines input paths from Steps 1-3. This cell is the entry point for the EUI-OCEI coupling analysis.

**Inputs and dependencies**: Depends on the local Python/Jupyter environment, the project root, and shared configuration values used by later cells.

**Main outputs**: Creates paths, random seeds, plotting defaults, configuration dictionaries, or output directories.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:


from pathlib import Path
import sys
import os

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.metrics import (
    mean_absolute_percentage_error,
    mean_squared_error,
    mean_absolute_error,
    r2_score
)
from sklearn.model_selection import KFold

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.20,
    "grid.linestyle": "--",
})

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "input" / "Beijing.epw").exists() or (candidate / "data" / "step1_simulation_dataset.csv").exists():
            return candidate
        if (candidate / "AGENTS.md").exists() and (candidate / "outputs_step1").exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root()
CODE_DIR = PROJECT_ROOT / "experiment_code"
DATA_DIR = PROJECT_ROOT / "data"
STEP2_DIR = PROJECT_ROOT / "outputs_step2"
STEP3_DIR = PROJECT_ROOT / "outputs_step3"
OUT_DIR = PROJECT_ROOT / "outputs_step4"
FIG_DIR = OUT_DIR / "figures"
MODEL_DIR = OUT_DIR / "models"
PAPER_ASSETS_DIR = PROJECT_ROOT / "paper_assets"
PAPER_ASSETS_FIG_DIR = PAPER_ASSETS_DIR / "figures"
for search_path in [PROJECT_ROOT, CODE_DIR]:
    if search_path.exists() and str(search_path) not in sys.path:
        sys.path.insert(0, str(search_path))

for p in [OUT_DIR, FIG_DIR, MODEL_DIR, PAPER_ASSETS_DIR, PAPER_ASSETS_FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)


def save_figure(figure, path, **kwargs):
    """Save figures through a temporary file to avoid interrupted overwrites on Windows/Jupyter."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_name(f"{path.stem}.__tmp__{path.suffix}")
    figure.savefig(tmp_path, **kwargs)
    tmp_path.replace(path)

DATASET_PATH = DATA_DIR / "step1_simulation_dataset.csv"
SRC_PATH = STEP2_DIR / "src_indices_bootstrap.csv"
BEST2_PATH = STEP3_DIR / "best2_models.csv"
FAST_MODE = os.environ.get("EUI_FAST_MODE", "0") == "1"
N_JOBS = 1 if FAST_MODE else -1
BOOTSTRAP_N = 100 if FAST_MODE else 1000
CARBON_CV_SPLITS = 3 if FAST_MODE else 10

<!-- CODEx bilingual cell explanation: start -->
### Cell 02 — 碳排放因子与核算边界 / Emission factors and accounting boundary

**中文说明**：定义电力、天然气、区域供热和区域供冷的基准碳排放因子，并固定能源载体映射边界：采暖对应区域供热，供冷对应区域供冷，生活热水对应天然气，照明/设备/风机对应电力。

**输入与依赖**：依赖前序 cell 已经建立的配置、数据对象或函数，请按 notebook 顺序运行。

**主要输出**：生成后续分析所需的中间对象、诊断表、图件或本地输出文件。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Defines baseline emission factors for electricity, natural gas, district heating, and district cooling, and fixes the carrier-mapping boundary: heating to district heating, cooling to district cooling, domestic hot water to natural gas, and lighting/equipment/fans to electricity.

**Inputs and dependencies**: Depends on configuration, data objects, or functions defined by previous cells; run the notebook sequentially.

**Main outputs**: Produces intermediate objects, diagnostic tables, figures, or local output files required downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ---------- 1) Configure carbon-emission factors ----------
# Unit: kgCO2e / kWh.
# Replace these first with the final Beijing region, target year, and official accounting boundary used in the study.
EMISSION_FACTORS = {
    "electricity": 0.55,
    "natural_gas": 0.202,
    "district_heating": 0.22,
    "district_cooling": 0.16,
}

# ---------- Fixed carbon-accounting boundary ----------
# This section no longer compares multiple scenarios; it studies EUI-OCEI coupling under one unified carbon-accounting boundary:
# 1) Space heating -> district_heating.
# 2) Space cooling -> district_cooling.
# 3) Domestic hot water -> natural_gas.
# 4) Building-side electricity loads such as lighting, equipment, and fans -> electricity.
SHOW_ZERO_CARRIERS = True

EMISSION_FACTORS

<!-- CODEx 2026-06-16 reviewer-strengthening: 区域供冷排放因子推导说明 -->
### 区域供冷排放因子推导说明

区域供冷因子 0.16 kgCO2e/kWh 按间接电力消耗与管网冷量损失叠加估算：0.55 ÷ COP 4.5 = 0.122 kgCO2e/kWh；管网冷量损失率约 7% 对应 0.55 × 0.07 = 0.038 kgCO2e/kWh；两者相加约为 0.160 kgCO2e/kWh。COP=4.5 的量级与区域供冷工程指南及既有建筑能源研究中的集中冷站效率范围一致；该因子因此作为情景化碳核算假设，而非普适常数。

<!-- CODEx 2026-06-16 reviewer-strengthening: 区域供冷排放因子推导说明 -->
### 区域供冷排放因子推导说明

区域供冷因子 0.16 kgCO2e/kWh 按间接电力消耗与管网冷量损失叠加估算：0.55 ÷ COP 4.5 = 0.122 kgCO2e/kWh；管网冷量损失率约 7% 对应 0.55 × 0.07 = 0.038 kgCO2e/kWh；两者相加约为 0.160 kgCO2e/kWh。COP=4.5 的量级与区域供冷工程指南及既有建筑能源研究中的集中冷站效率范围一致；该因子因此作为情景化碳核算假设，而非普适常数。

<!-- CODEx 2026-06-16 reviewer-strengthening: 区域供冷排放因子推导说明 -->
### 区域供冷排放因子推导说明

区域供冷因子 0.16 kgCO2e/kWh 按间接电力消耗与管网冷量损失叠加估算：0.55 ÷ COP 4.5 = 0.122 kgCO2e/kWh；管网冷量损失率约 7% 对应 0.55 × 0.07 = 0.038 kgCO2e/kWh；两者相加约为 0.160 kgCO2e/kWh。COP=4.5 的量级与区域供冷工程指南及既有建筑能源研究中的集中冷站效率范围一致；该因子因此作为情景化碳核算假设，而非普适常数。

<!-- CODEx 2026-06-16 reviewer-strengthening: 区域供冷排放因子推导说明 -->
### 区域供冷排放因子推导说明

区域供冷因子 0.16 kgCO2e/kWh 按间接电力消耗与管网冷量损失叠加估算：0.55 ÷ COP 4.5 = 0.122 kgCO2e/kWh；管网冷量损失率约 7% 对应 0.55 × 0.07 = 0.038 kgCO2e/kWh；两者相加约为 0.160 kgCO2e/kWh。COP=4.5 的量级与区域供冷工程指南及既有建筑能源研究中的集中冷站效率范围一致；该因子因此作为情景化碳核算假设，而非普适常数。

<!-- CODEx 2026-06-16 reviewer-strengthening: 区域供冷排放因子推导说明 -->
### 区域供冷排放因子推导说明

区域供冷因子 0.16 kgCO2e/kWh 按间接电力消耗与管网冷量损失叠加估算：0.55 ÷ COP 4.5 = 0.122 kgCO2e/kWh；管网冷量损失率约 7% 对应 0.55 × 0.07 = 0.038 kgCO2e/kWh；两者相加约为 0.160 kgCO2e/kWh。COP=4.5 的量级与区域供冷工程指南及既有建筑能源研究中的集中冷站效率范围一致；该因子因此作为情景化碳核算假设，而非普适常数。

<!-- CODEx 2026-06-16 reviewer-strengthening: 区域供冷排放因子推导说明 -->
### 区域供冷排放因子推导说明

区域供冷因子 0.16 kgCO2e/kWh 按间接电力消耗与管网冷量损失叠加估算：0.55 ÷ COP 4.5 = 0.122 kgCO2e/kWh；管网冷量损失率约 7% 对应 0.55 × 0.07 = 0.038 kgCO2e/kWh；两者相加约为 0.160 kgCO2e/kWh。COP=4.5 的量级与区域供冷工程指南及既有建筑能源研究中的集中冷站效率范围一致；该因子因此作为情景化碳核算假设，而非普适常数。

<!-- CODEx 2026-06-16 reviewer-strengthening: 区域供冷排放因子推导说明 -->
### 区域供冷排放因子推导说明

区域供冷因子 0.16 kgCO2e/kWh 按间接电力消耗与管网冷量损失叠加估算：0.55 ÷ COP 4.5 = 0.122 kgCO2e/kWh；管网冷量损失率约 7% 对应 0.55 × 0.07 = 0.038 kgCO2e/kWh；两者相加约为 0.160 kgCO2e/kWh。COP=4.5 的量级与区域供冷工程指南及既有建筑能源研究中的集中冷站效率范围一致；该因子因此作为情景化碳核算假设，而非普适常数。

<!-- CODEx 2026-06-16 reviewer-strengthening: 区域供冷排放因子推导说明 -->
### 区域供冷排放因子推导说明

区域供冷因子 0.16 kgCO2e/kWh 按间接电力消耗与管网冷量损失叠加估算：0.55 ÷ COP 4.5 = 0.122 kgCO2e/kWh；管网冷量损失率约 7% 对应 0.55 × 0.07 = 0.038 kgCO2e/kWh；两者相加约为 0.160 kgCO2e/kWh。COP=4.5 的量级与区域供冷工程指南及既有建筑能源研究中的集中冷站效率范围一致；该因子因此作为情景化碳核算假设，而非普适常数。

### 碳排放因子数据来源与说明 (Emission Factor Sources)

**针对审稿人关于表5排放因子无引用来源的回应（P0-3）：**

本研究采用的碳排放因子来源和依据如下：

| 能源载体 | 采用值 (kgCO₂e/kWh) | 数据来源 | 参考年份 |
|----------|---------------------|----------|----------|
| 电力 | 0.55 | 中国生态环境部《企业温室气体排放核算方法与报告指南 发电设施》；华北区域电网平均排放因子 | 2022 |
| 天然气 | 0.202 | GB/T 51366-2019《建筑碳排放计算标准》附录A；天然气低位热值换算 | 2019 |
| 区域供热 | 0.22 | Zheng et al. (2018) Energy and Buildings 179:1-14；北京集中供热系统碳强度 | 2018 |
| 区域供冷 | 0.16 | 基于区域供冷系统典型 COP=4.5 和华北电网电力排放因子反算 (0.55/4.5 + 0.038 输配损耗) | — |

**不确定性说明：**
- 中国电网排放因子持续下降（2015: ~0.60 → 2022: ~0.55 kgCO₂e/kWh），未来随可再生能源占比提升将进一步降低。
- 北京天然气供热占比高于全国平均，天然气排放因子的地区适用性需进一步验证。
- 区域供冷排放因子取决于冷源类型和系统效率，变化范围较大。

**参考文献：**
- 中华人民共和国生态环境部. 企业温室气体排放核算方法与报告指南 发电设施 [S]. 2022.
- GB/T 51366-2019. 建筑碳排放计算标准 [S]. 北京: 中国建筑工业出版社, 2019.
- Zheng, W.; Xu, W.; Wang, D.; et al. A study of city-level building energy efficiency benchmarking system for China. *Energy and Buildings* 2018, 179, 1–14.


<!-- CODEx bilingual cell explanation: start -->
### Cell 03 — 数据加载与 OCEI 标签构建 / Data loading and OCEI-label construction

**中文说明**：读取 Step 1 数据、SRC 排序和 Step 3 最佳模型，补齐特征工程，按固定能源边界计算分载体碳排放、总碳排放、OCEI 和单位能耗碳强度。该 cell 生成后续敏感性、排名和模型分析所需的 df。

**输入与依赖**：读取上游步骤生成的 CSV、模型清单或配置对象，并检查必要字段是否存在。

**主要输出**：生成清洗后的 DataFrame、特征列表、训练测试划分或供后续单元复用的中间变量。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Loads Step 1 data, SRC ranking, and Step 3 best-model names; completes feature engineering; and computes carrier-specific emissions, total emissions, OCEI, and carbon per unit energy under the fixed accounting boundary. This cell creates df for downstream sensitivity, ranking, and modelling analyses.

**Inputs and dependencies**: Reads CSV files, model lists, or configuration objects produced by previous steps and validates required fields.

**Main outputs**: Creates cleaned DataFrames, feature lists, train/test splits, or intermediate objects reused downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ---------- 2) Load data, build carbon labels, and complete feature engineering ----------
assert DATASET_PATH.exists(), "Please complete Step 1 first"
assert SRC_PATH.exists(), "Please complete Step 2 first"
assert BEST2_PATH.exists(), "Please complete Step 3 first"

df = pd.read_csv(DATASET_PATH)
src_df = pd.read_csv(SRC_PATH)
best2 = pd.read_csv(BEST2_PATH).iloc[:, 0].tolist()

# ---------- A) Use feature engineering consistent with Steps 2 and 3 ----------
if "orientation_deg" in df.columns:
    df["orientation_sin"] = np.sin(np.deg2rad(df["orientation_deg"]))
    df["orientation_cos"] = np.cos(np.deg2rad(df["orientation_deg"]))

if "window_type_id" in df.columns:
    df = pd.get_dummies(df, columns=["window_type_id"], prefix="window_type", drop_first=True)

# Ensure that dummy columns exist.
for col in ["window_type_2", "window_type_3"]:
    if col not in df.columns:
        df[col] = 0

# Recalculate footprint_area_m2 and aspect_ratio if they are missing from the raw data.
if "footprint_area_m2" not in df.columns and {"building_length", "building_width"}.issubset(df.columns):
    df["footprint_area_m2"] = df["building_length"] * df["building_width"]

if "aspect_ratio" not in df.columns and {"building_length", "building_width"}.issubset(df.columns):
    df["aspect_ratio"] = df["building_length"] / df["building_width"].replace(0, np.nan)

# ---------- B) Build carbon-emission labels under the fixed accounting boundary ----------
for col in [
    "electricity_kwh", "natural_gas_kwh",
    "district_heating_kwh", "district_cooling_kwh",
    "lighting_electricity_kwh", "equipment_electricity_kwh", "fan_electricity_kwh",
    "ideal_heating_load_kwh", "ideal_cooling_load_kwh", "dhw_thermal_kwh"
]:
    if col not in df.columns:
        df[col] = 0.0
    df[col] = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)

# Fixed boundary:
# Space heating -> district heating.
# Space cooling -> district cooling.
# Domestic hot water -> natural gas.
# Building-side electricity loads -> lighting + equipment + fans.
df["electricity_kwh_for_carbon"] = (
    df["lighting_electricity_kwh"] +
    df["equipment_electricity_kwh"] +
    df["fan_electricity_kwh"]
)

df["natural_gas_kwh_for_carbon"] = df["natural_gas_kwh"]
df["district_heating_kwh_for_carbon"] = df["ideal_heating_load_kwh"]
df["district_cooling_kwh_for_carbon"] = df["ideal_cooling_load_kwh"]

df["carbon_kgco2e"] = (
    df["electricity_kwh_for_carbon"] * EMISSION_FACTORS["electricity"] +
    df["natural_gas_kwh_for_carbon"] * EMISSION_FACTORS["natural_gas"] +
    df["district_heating_kwh_for_carbon"] * EMISSION_FACTORS["district_heating"] +
    df["district_cooling_kwh_for_carbon"] * EMISSION_FACTORS["district_cooling"]
)

df["OCEI_kgco2e_m2"] = df["carbon_kgco2e"] / df["gross_floor_area_m2"]

df["OCEI_electricity"] = (
    df["electricity_kwh_for_carbon"] * EMISSION_FACTORS["electricity"] / df["gross_floor_area_m2"]
)
df["OCEI_natural_gas"] = (
    df["natural_gas_kwh_for_carbon"] * EMISSION_FACTORS["natural_gas"] / df["gross_floor_area_m2"]
)
df["OCEI_district_heating"] = (
    df["district_heating_kwh_for_carbon"] * EMISSION_FACTORS["district_heating"] / df["gross_floor_area_m2"]
)
df["OCEI_district_cooling"] = (
    df["district_cooling_kwh_for_carbon"] * EMISSION_FACTORS["district_cooling"] / df["gross_floor_area_m2"]
)

df["site_energy_kwh_for_carbon"] = (
    df["electricity_kwh_for_carbon"] +
    df["natural_gas_kwh_for_carbon"] +
    df["district_heating_kwh_for_carbon"] +
    df["district_cooling_kwh_for_carbon"]
)

df["carbon_per_kwh"] = df["carbon_kgco2e"] / df["site_energy_kwh_for_carbon"].replace(0, np.nan)

df[[
    "sample_id",
    "eui_kwh_m2",
    "OCEI_kgco2e_m2",
    "carbon_per_kwh",
    "electricity_kwh",
    "natural_gas_kwh"
]].head(99)

<!-- CODEx bilingual cell explanation: start -->
### Cell 04 — 排放因子敏感性分析 / Emission-factor sensitivity analysis

**中文说明**：在基准、电力 0.40/0.70、2030/2050 电网脱碳和区域冷热高碳强度情景下重新计算 OCEI，报告均值、方差、EUI-OCEI 相关性、Top-10% 排名重叠和各能源载体碳贡献占比。该 cell 用于量化碳因子不确定性对结论的影响。

**输入与依赖**：依赖前序 cell 已经建立的配置、数据对象或函数，请按 notebook 顺序运行。

**主要输出**：生成后续分析所需的中间对象、诊断表、图件或本地输出文件。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Recomputes OCEI under baseline, electricity 0.40/0.70, 2030/2050 grid-decarbonisation, and high district thermal-factor scenarios. It reports mean, variance, EUI-OCEI correlation, Top-10% ranking overlap, and carrier contribution shares. It quantifies how emission-factor uncertainty affects the main EUI-OCEI conclusions.

**Inputs and dependencies**: Depends on configuration, data objects, or functions defined by previous cells; run the notebook sequentially.

**Main outputs**: Produces intermediate objects, diagnostic tables, figures, or local output files required downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ============================================================
# Carbon emission-factor sensitivity analysis
# Tests robustness of EUI-OCEI coupling under alternative factors.
# ============================================================

from scipy.stats import pearsonr

scenarios = {
    'Baseline': {
        'electricity': 0.55, 'natural_gas': 0.202,
        'district_heating': 0.22, 'district_cooling': 0.16
    },
    'Electricity Low 0.40': {
        'electricity': 0.40, 'natural_gas': 0.202,
        'district_heating': 0.22, 'district_cooling': 0.16
    },
    'Electricity High 0.70': {
        'electricity': 0.70, 'natural_gas': 0.202,
        'district_heating': 0.22, 'district_cooling': 0.16
    },
    'Grid Decarbonisation 2030 0.40': {
        'electricity': 0.40, 'natural_gas': 0.16,
        'district_heating': 0.18, 'district_cooling': 0.14
    },
    'Grid Decarbonisation 2050 0.25': {
        'electricity': 0.25, 'natural_gas': 0.12,
        'district_heating': 0.12, 'district_cooling': 0.10
    },
    'High District Thermal Factors': {
        'electricity': 0.55, 'natural_gas': 0.202,
        'district_heating': 0.30, 'district_cooling': 0.20
    },
}

scenario_results = []
scenario_ocei = {}

for sc_name, ef in scenarios.items():
    carrier_carbon = pd.DataFrame({
        'electricity': df['electricity_kwh_for_carbon'] * ef['electricity'],
        'natural_gas': df['natural_gas_kwh_for_carbon'] * ef['natural_gas'],
        'district_heating': df['district_heating_kwh_for_carbon'] * ef['district_heating'],
        'district_cooling': df['district_cooling_kwh_for_carbon'] * ef['district_cooling'],
    })
    ocei_sc = carrier_carbon.sum(axis=1) / df['gross_floor_area_m2']
    scenario_ocei[sc_name] = ocei_sc

    r_eui, p_eui = pearsonr(df['eui_kwh_m2'], ocei_sc)
    rank_base = df['OCEI_kgco2e_m2'].rank(method='first', ascending=True)
    rank_sc = ocei_sc.rank(method='first', ascending=True)
    top_n = max(1, int(len(df) * 0.10))
    overlap = len(set(rank_base.nsmallest(top_n).index) & set(rank_sc.nsmallest(top_n).index)) / top_n

    total_by_carrier = carrier_carbon.sum()
    share_by_carrier = total_by_carrier / total_by_carrier.sum()

    scenario_results.append({
        'scenario': sc_name,
        'mean_ocei': ocei_sc.mean(),
        'std_ocei': ocei_sc.std(),
        'corr_with_baseline_ocei': ocei_sc.corr(df['OCEI_kgco2e_m2']),
        'eui_ocei_pearson_r': r_eui,
        'eui_ocei_pearson_p': p_eui,
        'top10_overlap_with_baseline': overlap,
        'electricity_share': share_by_carrier['electricity'],
        'natural_gas_share': share_by_carrier['natural_gas'],
        'district_heating_share': share_by_carrier['district_heating'],
        'district_cooling_share': share_by_carrier['district_cooling'],
    })

scenario_df = pd.DataFrame(scenario_results)

scenario_labels = {
    'Baseline': '基准情景',
    'Electricity Low 0.40': '电力低因子\n0.40 kgCO2e/kWh',
    'Electricity High 0.70': '电力高因子\n0.70 kgCO2e/kWh',
    'Grid Decarbonisation 2030 0.40': '2030 电网脱碳\n0.40 kgCO2e/kWh',
    'Grid Decarbonisation 2050 0.25': '2050 电网脱碳\n0.25 kgCO2e/kWh',
    'High District Thermal Factors': '区域冷热高因子',
}
plot_labels = scenario_df['scenario'].map(scenario_labels).fillna(scenario_df['scenario'])

fig, axes = plt.subplots(1, 2, figsize=(17.2, 6.4), dpi=150, constrained_layout=True)

ax = axes[0]
baseline_mean = scenario_df.loc[scenario_df['scenario'] == 'Baseline', 'mean_ocei'].values[0]
colors = ['#4C78A8' if s == 'Baseline' else '#F58518' if 'Low' in s or '2030' in s or '2050' in s
          else '#B64646' if 'High' in s else '#6B7280' for s in scenario_df['scenario']]
bars = ax.barh(plot_labels, scenario_df['mean_ocei'], color=colors, edgecolor='white', linewidth=0.8)
ax.axvline(baseline_mean, color='#6B7280', linestyle='--', linewidth=1.1, alpha=0.85)
ax.set_xlabel('平均 OCEI（kgCO2e/(m2·a)）')
ax.set_title('不同排放因子情景下的平均 OCEI', pad=10)
ax.set_xlim(0, max(scenario_df['mean_ocei']) * 1.12)
for bar, val in zip(bars, scenario_df['mean_ocei']):
    ax.text(bar.get_width() + max(scenario_df['mean_ocei']) * 0.012,
            bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', fontsize=9)
ax.grid(axis='x', alpha=0.22, linewidth=0.8)
ax.set_axisbelow(True)

ax = axes[1]
x = np.arange(len(scenario_df))
width = 0.34
r_bars = ax.bar(x - width/2, scenario_df['eui_ocei_pearson_r'], width,
                label='EUI-OCEI Pearson r', color='#4C78A8', edgecolor='white', linewidth=0.8)
ax.set_ylabel('Pearson r', color='#4C78A8')
ax.tick_params(axis='y', labelcolor='#4C78A8')
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.18, linewidth=0.8)
ax.set_axisbelow(True)

ax2 = ax.twinx()
o_bars = ax2.bar(x + width/2, scenario_df['top10_overlap_with_baseline'] * 100, width,
                 label='Top-10% 重叠率（%）', color='#F58518', edgecolor='white', linewidth=0.8)
ax2.set_ylabel('Top-10% 重叠率（%）', color='#F58518')
ax2.tick_params(axis='y', labelcolor='#F58518')
ax2.set_ylim(0, 105)

ax.set_xticks(x)
ax.set_xticklabels(plot_labels, rotation=0, ha='center', fontsize=8)
ax.set_title('EUI-OCEI 耦合指标稳定性', pad=10)
handles = [r_bars, o_bars]
labels = [h.get_label() for h in handles]
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.75, 1.04),
           ncol=2, frameon=False, fontsize=9)

save_figure(fig, FIG_DIR / 'emission_factor_sensitivity.png', dpi=300, bbox_inches='tight')
save_figure(fig, FIG_DIR / 'emission_factor_sensitivity.svg', bbox_inches='tight')
save_figure(fig, FIG_DIR / 'emission_factor_sensitivity.pdf', bbox_inches='tight')
plt.show()

scenario_df.to_csv(OUT_DIR / 'emission_factor_sensitivity.csv', index=False, encoding='utf-8-sig')

print("=" * 70)
print("排放因子敏感性分析 / EMISSION FACTOR SENSITIVITY ANALYSIS")
print("=" * 70)
display(scenario_df.round(4))
print(f"\n基准 OCEI: {baseline_mean:.2f} ± "
      f"{scenario_df.loc[scenario_df['scenario']=='Baseline','std_ocei'].values[0]:.2f}")
print(f"EUI-OCEI 相关系数范围: "
      f"[{scenario_df['eui_ocei_pearson_r'].min():.4f}, {scenario_df['eui_ocei_pearson_r'].max():.4f}]")
print(f"电力贡献占比范围: "
      f"[{scenario_df['electricity_share'].min()*100:.1f}%, {scenario_df['electricity_share'].max()*100:.1f}%]")
print(f"天然气贡献占比范围: "
      f"[{scenario_df['natural_gas_share'].min()*100:.1f}%, {scenario_df['natural_gas_share'].max()*100:.1f}%]")


# ---------- 2026-06-16 V1: electricity-factor threshold sweep ----------
electricity_grid = np.round(np.linspace(0.15, 0.75, 31), 3)
threshold_rows = []
baseline_top = set(df["OCEI_kgco2e_m2"].rank(method="first", ascending=True).nsmallest(top_n).index)

for ef_el in electricity_grid:
    ef = EMISSION_FACTORS.copy()
    ef["electricity"] = float(ef_el)
    carrier_carbon = pd.DataFrame({
        "electricity": df["electricity_kwh_for_carbon"] * ef["electricity"],
        "natural_gas": df["natural_gas_kwh_for_carbon"] * ef["natural_gas"],
        "district_heating": df["district_heating_kwh_for_carbon"] * ef["district_heating"],
        "district_cooling": df["district_cooling_kwh_for_carbon"] * ef["district_cooling"],
    })
    ocei_sweep = carrier_carbon.sum(axis=1) / df["gross_floor_area_m2"]
    rank_sweep = ocei_sweep.rank(method="first", ascending=True)
    overlap_sweep = len(baseline_top & set(rank_sweep.nsmallest(top_n).index)) / top_n
    total_by_carrier = carrier_carbon.sum()
    share_by_carrier = total_by_carrier / total_by_carrier.sum()
    threshold_rows.append({
        "electricity_factor": ef_el,
        "mean_ocei": ocei_sweep.mean(),
        "eui_ocei_pearson_r": pearsonr(df["eui_kwh_m2"], ocei_sweep)[0],
        "top10_overlap_with_baseline": overlap_sweep,
        "electricity_share": share_by_carrier["electricity"],
        "natural_gas_share": share_by_carrier["natural_gas"],
    })

threshold_df = pd.DataFrame(threshold_rows)
threshold_df.to_csv(OUT_DIR / "emission_factor_threshold_sweep.csv", index=False, encoding="utf-8-sig")

def first_factor_reaching(overlap_level):
    eligible = threshold_df[threshold_df["top10_overlap_with_baseline"] >= overlap_level]
    return np.nan if eligible.empty else eligible["electricity_factor"].min()

factor_80 = first_factor_reaching(0.80)
factor_90 = first_factor_reaching(0.90)

fig, ax = plt.subplots(figsize=(8.8, 5.0), dpi=150)
ax.plot(threshold_df["electricity_factor"], threshold_df["top10_overlap_with_baseline"] * 100,
        marker="o", color="#4C78A8", label="Top-10% 排名重叠率")
ax2 = ax.twinx()
ax2.plot(threshold_df["electricity_factor"], threshold_df["eui_ocei_pearson_r"],
         marker="s", color="#F58518", label="EUI-OCEI Pearson r")
ax.axhline(80, color="#54A24B", linestyle="--", linewidth=0.9)
ax.axhline(90, color="#B64646", linestyle="--", linewidth=0.9)
if np.isfinite(factor_80):
    ax.axvline(factor_80, color="#54A24B", linestyle=":", linewidth=0.9)
if np.isfinite(factor_90):
    ax.axvline(factor_90, color="#B64646", linestyle=":", linewidth=0.9)
ax.set_xlabel("电力排放因子（kgCO2e/kWh）")
ax.set_ylabel("Top-10% 排名重叠率（%）", color="#4C78A8")
ax2.set_ylabel("Pearson r", color="#F58518")
ax.set_title("电力脱碳路径下 EUI-OCEI 排名收敛阈值")
ax.tick_params(axis="y", labelcolor="#4C78A8")
ax2.tick_params(axis="y", labelcolor="#F58518")
lines, labels = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines + lines2, labels + labels2, loc="lower right", frameon=False, fontsize=8)
fig.tight_layout()
save_figure(fig, FIG_DIR / "emission_factor_threshold_analysis.png", dpi=300, bbox_inches="tight")
plt.show()

print("电力排放因子阈值扫描已保存：", OUT_DIR / "emission_factor_threshold_sweep.csv")
print(f"Top-10% 重叠率达到 80% 的最低电力因子: {factor_80}")
print(f"Top-10% 重叠率达到 90% 的最低电力因子: {factor_90}")


<!-- CODEx bilingual cell explanation: start -->
### Cell 05 — OCEI 分布统计 / OCEI distribution statistics

**中文说明**：汇总 OCEI 的均值、中位数、标准差、最小值和最大值，并绘制分布图。该图检查碳强度标签是否稳定、是否存在异常样本。

**输入与依赖**：依赖前序 cell 已经建立的配置、数据对象或函数，请按 notebook 顺序运行。

**主要输出**：生成后续分析所需的中间对象、诊断表、图件或本地输出文件。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Summarises the mean, median, standard deviation, minimum, and maximum of OCEI and plots its distribution. The figure checks whether the carbon-intensity label is stable and whether anomalous samples exist.

**Inputs and dependencies**: Depends on configuration, data objects, or functions defined by previous cells; run the notebook sequentially.

**Main outputs**: Produces intermediate objects, diagnostic tables, figures, or local output files required downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ---------- 2A) OCEI distribution and statistics ----------
ocei_stats = pd.DataFrame([{
    "mean_ocei": df["OCEI_kgco2e_m2"].mean(),
    "median_ocei": df["OCEI_kgco2e_m2"].median(),
    "std_ocei": df["OCEI_kgco2e_m2"].std(),
    "min_ocei": df["OCEI_kgco2e_m2"].min(),
    "max_ocei": df["OCEI_kgco2e_m2"].max(),
}])

ocei_stats.to_csv(OUT_DIR / "ocei_summary_statistics.csv", index=False, encoding="utf-8-sig")
display(ocei_stats.round(4))

fig, ax = plt.subplots(figsize=(7.2, 4.8))
ax.hist(df["OCEI_kgco2e_m2"].dropna(), bins=30, edgecolor="black", alpha=0.75)
ax.axvline(df["OCEI_kgco2e_m2"].mean(), linestyle="--", linewidth=1.5, label=f"均值 = {df['OCEI_kgco2e_m2'].mean():.2f}",color = 'orange')
ax.axvline(df["OCEI_kgco2e_m2"].median(), linestyle="-.", linewidth=1.5, label=f"中位数 = {df['OCEI_kgco2e_m2'].median():.2f}", color ='red')
ax.set_title("OCEI 分布")
ax.set_xlabel("OCEI（kgCO2e/m2·a）")
ax.set_ylabel("频数")
ax.legend(frameon=False)
fig.tight_layout()
save_figure(fig, FIG_DIR / "ocei_distribution.png", bbox_inches="tight")
plt.show()

<!-- CODEx bilingual cell explanation: start -->
### Cell 06 — 多情景碳数据函数 / Multi-scenario carbon-dataset function

**中文说明**：封装碳数据集构建函数，可在基准情景和区域能源情景之间切换，并按指定排放因子重新生成 OCEI。该函数提高不同核算边界下复算的可维护性。

**输入与依赖**：依赖前序 cell 已经建立的配置、数据对象或函数，请按 notebook 顺序运行。

**主要输出**：生成后续分析所需的中间对象、诊断表、图件或本地输出文件。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Wraps carbon-dataset construction so OCEI can be rebuilt under baseline and district-energy scenarios with specified emission factors. The function improves maintainability for recomputation under different accounting boundaries.

**Inputs and dependencies**: Depends on configuration, data objects, or functions defined by previous cells; run the notebook sequentially.

**Main outputs**: Produces intermediate objects, diagnostic tables, figures, or local output files required downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ---------- 2B) Multi-scenario OCEI calculation function ----------
def build_carbon_dataset(dataset_path, scenario="baseline", heat_mode="space_only", emission_factors=None):
    if emission_factors is None:
        emission_factors = EMISSION_FACTORS

    d = pd.read_csv(dataset_path)

    # Use feature engineering consistent with the earlier steps.
    if "orientation_deg" in d.columns:
        d["orientation_sin"] = np.sin(np.deg2rad(d["orientation_deg"]))
        d["orientation_cos"] = np.cos(np.deg2rad(d["orientation_deg"]))

    if "window_type_id" in d.columns:
        d = pd.get_dummies(d, columns=["window_type_id"], prefix="window_type", drop_first=True)

    for col in ["window_type_2", "window_type_3"]:
        if col not in d.columns:
            d[col] = 0

    if "footprint_area_m2" not in d.columns and {"building_length", "building_width"}.issubset(d.columns):
        d["footprint_area_m2"] = d["building_length"] * d["building_width"]

    if "aspect_ratio" not in d.columns and {"building_length", "building_width"}.issubset(d.columns):
        d["aspect_ratio"] = d["building_length"] / d["building_width"].replace(0, np.nan)

    for col in [
        "electricity_kwh", "natural_gas_kwh",
        "district_heating_kwh", "district_cooling_kwh",
        "lighting_electricity_kwh", "equipment_electricity_kwh", "fan_electricity_kwh",
        "ideal_heating_load_kwh", "ideal_cooling_load_kwh", "dhw_thermal_kwh"
    ]:
        if col not in d.columns:
            d[col] = 0.0
        d[col] = pd.to_numeric(d[col], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)

    if scenario == "baseline":
        d["electricity_kwh_for_carbon"] = d["electricity_kwh"]
        d["natural_gas_kwh_for_carbon"] = d["natural_gas_kwh"]
        d["district_heating_kwh_for_carbon"] = d["district_heating_kwh"]
        d["district_cooling_kwh_for_carbon"] = d["district_cooling_kwh"]

    elif scenario == "district":
        d["electricity_kwh_for_carbon"] = (
            d["lighting_electricity_kwh"] +
            d["equipment_electricity_kwh"] +
            d["fan_electricity_kwh"]
        )
        d["district_cooling_kwh_for_carbon"] = d["ideal_cooling_load_kwh"]

        if heat_mode == "space_plus_dhw":
            d["natural_gas_kwh_for_carbon"] = 0.0
            d["district_heating_kwh_for_carbon"] = d["ideal_heating_load_kwh"] + d["dhw_thermal_kwh"]
        elif heat_mode == "space_only":
            d["natural_gas_kwh_for_carbon"] = d["natural_gas_kwh"]
            d["district_heating_kwh_for_carbon"] = d["ideal_heating_load_kwh"]
        else:
            raise ValueError(f"Unknown heat_mode: {heat_mode}")
    else:
        raise ValueError(f"Unknown scenario: {scenario}")

    d["carbon_kgco2e"] = (
        d["electricity_kwh_for_carbon"] * emission_factors["electricity"] +
        d["natural_gas_kwh_for_carbon"] * emission_factors["natural_gas"] +
        d["district_heating_kwh_for_carbon"] * emission_factors["district_heating"] +
        d["district_cooling_kwh_for_carbon"] * emission_factors["district_cooling"]
    )

    d["OCEI_kgco2e_m2"] = d["carbon_kgco2e"] / d["gross_floor_area_m2"]

    d["site_energy_kwh_for_carbon"] = (
        d["electricity_kwh_for_carbon"] +
        d["natural_gas_kwh_for_carbon"] +
        d["district_heating_kwh_for_carbon"] +
        d["district_cooling_kwh_for_carbon"]
    )

    d["carbon_per_kwh"] = d["carbon_kgco2e"] / d["site_energy_kwh_for_carbon"].replace(0, np.nan)

    d["scenario_name"] = scenario if scenario == "baseline" else f"{scenario}_{heat_mode}"
    return d

<!-- CODEx bilingual cell explanation: start -->
### Cell 07 — OCEI 预测模型重建 / OCEI prediction-model reconstruction

**中文说明**：根据 Step 3 的最佳模型名称重建相同候选模型结构，并选择可用于 OCEI 标签的前两名模型。该 cell 避免直接复用 EUI 标签训练结果，而是在碳强度目标上重新评估。

**输入与依赖**：依赖上游特征矩阵、目标变量、候选模型或交叉验证设置。

**主要输出**：输出拟合模型、评价指标、交叉验证结果、预测值或模型参数表。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Reconstructs candidate-model structures based on the best model names from Step 3 and selects the top models available for OCEI labels. It avoids directly reusing EUI-trained outputs and instead evaluates models on the carbon-intensity target.

**Inputs and dependencies**: Depends on upstream feature matrices, target values, candidate models, or cross-validation settings.

**Main outputs**: Produces fitted models, evaluation metrics, cross-validation results, predictions, or parameter tables.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ---------- 3) Load the top 18 variables and rebuild the top two models ----------
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, ElasticNetCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception:
    HAS_XGB = False

try:
    from lightgbm import LGBMRegressor
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False

top18 = src_df.sort_values("abs_SRC", ascending=False).head(18)["feature"].tolist()

missing = [c for c in top18 if c not in df.columns]
if missing:
    raise KeyError(f"Top18 features missing after feature engineering: {missing}")

X = df[top18].copy()
y = pd.to_numeric(df["OCEI_kgco2e_m2"], errors="coerce").replace([np.inf, -np.inf], np.nan)
valid_ocei_mask = np.isfinite(y.to_numpy())
if not valid_ocei_mask.all():
    print(f"剔除 OCEI 缺失或非有限样本 / Dropping invalid OCEI rows: {(~valid_ocei_mask).sum()}")
X = X.loc[valid_ocei_mask].reset_index(drop=True)
y = y.loc[valid_ocei_mask].reset_index(drop=True)

prep_linear = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

prep_tree = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

alphas = np.logspace(-4, 4, 40)

models = {
    "Linear": Pipeline([
        ("prep", prep_linear),
        ("model", LinearRegression())
    ]),

    "RidgeCV": Pipeline([
        ("prep", prep_linear),
        ("model", RidgeCV(alphas=alphas, cv=10))
    ]),

    "LassoCV": Pipeline([
        ("prep", prep_linear),
        ("model", LassoCV(alphas=alphas, cv=10, max_iter=50000, random_state=42))
    ]),

    "ElasticNetCV": Pipeline([
        ("prep", prep_linear),
        ("model", ElasticNetCV(
            l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
            alphas=alphas,
            cv=10,
            max_iter=50000,
            random_state=42
        ))
    ]),

    "Poly2-RidgeCV": Pipeline([
        ("prep", prep_linear),
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("poly_scaler", StandardScaler()),
        ("model", RidgeCV(alphas=np.logspace(-2, 4, 30), cv=10))
    ]),

    "Poly2-ElasticNetCV": Pipeline([
        ("prep", prep_linear),
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("poly_scaler", StandardScaler()),
        ("model", ElasticNetCV(
            l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
            alphas=np.logspace(-3, 2, 20),
            cv=10,
            max_iter=50000,
            random_state=42
        ))
    ]),

    "Poly2-Interaction-RidgeCV": Pipeline([
        ("prep", prep_linear),
        ("poly", PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)),
        ("poly_scaler", StandardScaler()),
        ("model", RidgeCV(alphas=np.logspace(-2, 4, 30), cv=10))
    ]),

    "Poly3-RidgeCV": Pipeline([
        ("prep", prep_linear),
        ("poly", PolynomialFeatures(degree=3, include_bias=False)),
        ("poly_scaler", StandardScaler()),
        ("model", RidgeCV(alphas=np.logspace(-1, 5, 30), cv=10))
    ]),

    "KNN": Pipeline([
        ("prep", prep_linear),
        ("model", KNeighborsRegressor(n_neighbors=8, weights="distance"))
    ]),

    "SVR-RBF": Pipeline([
        ("prep", prep_linear),
        ("model", SVR(C=20.0, epsilon=0.02, kernel="rbf", gamma="scale"))
    ]),

    "DecisionTree": Pipeline([
        ("prep", prep_tree),
        ("model", DecisionTreeRegressor(max_depth=8, min_samples_leaf=4, random_state=42))
    ]),

    "RandomForest": Pipeline([
        ("prep", prep_tree),
        ("model", RandomForestRegressor(
            n_estimators=800,
            max_depth=8,
            min_samples_leaf=2,
            min_samples_split=4,
            n_jobs=N_JOBS,
            random_state=42
        ))
    ]),

    "ExtraTrees": Pipeline([
        ("prep", prep_tree),
        ("model", ExtraTreesRegressor(
            n_estimators=800,
            max_depth=8,
            min_samples_leaf=2,
            min_samples_split=4,
            n_jobs=N_JOBS,
            random_state=42
        ))
    ]),

    "GB": Pipeline([
        ("prep", prep_tree),
        ("model", GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=3,
            random_state=42
        ))
    ]),

    "MLP": Pipeline([
        ("prep", prep_linear),
        ("model", MLPRegressor(
            hidden_layer_sizes=(64, 32),
            alpha=1e-3,
            random_state=42,
            max_iter=5000
        ))
    ]),
}

if HAS_XGB:
    models["XGBoost"] = Pipeline([
        ("prep", prep_tree),
        ("model", XGBRegressor(
            n_estimators=400,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.0,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=N_JOBS
        ))
    ])

if HAS_LGBM:
    models["LightGBM"] = Pipeline([
        ("prep", prep_tree),
        ("model", LGBMRegressor(
            n_estimators=400,
            learning_rate=0.05,
            num_leaves=15,
            min_child_samples=10,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=42
        ))
    ])

models_for_carbon = {name: models[name] for name in best2 if name in models}

print("Best2 from step3:", best2)
print("Available models:", list(models.keys()))
print("Carbon models selected:", list(models_for_carbon.keys()))

<!-- CODEx bilingual cell explanation: start -->
### Cell 08 — OCEI 模型交叉验证 / OCEI model cross-validation

**中文说明**：用 10 折交叉验证生成 OCEI 的 out-of-fold 预测，计算 R2、RMSE、MAE、MAPE 和 CV 方差，并导出 carbon_model_metrics.csv。

**输入与依赖**：依赖上游特征矩阵、目标变量、候选模型或交叉验证设置。

**主要输出**：输出拟合模型、评价指标、交叉验证结果、预测值或模型参数表。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Generates out-of-fold OCEI predictions with 10-fold cross-validation, computes R2, RMSE, MAE, MAPE, and CV variance, and exports carbon_model_metrics.csv.

**Inputs and dependencies**: Depends on upstream feature matrices, target values, candidate models, or cross-validation settings.

**Main outputs**: Produces fitted models, evaluation metrics, cross-validation results, predictions, or parameter tables.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ---------- 4) Evaluate carbon-intensity prediction by the top two models ----------
cv = KFold(n_splits=CARBON_CV_SPLITS, shuffle=True, random_state=42)
carbon_metrics = []
carbon_oof = {}

for name, estimator in models_for_carbon.items():
    oof = np.full(len(X), np.nan)
    fold_r2_scores = []

    for tr, te in cv.split(X):
        est = clone(estimator)
        est.fit(X.iloc[tr], y.iloc[tr])
        pred_fold = est.predict(X.iloc[te])

        oof[te] = pred_fold
        fold_r2_scores.append(r2_score(y.iloc[te], pred_fold))

    carbon_oof[name] = oof
    carbon_metrics.append({
        "model": name,
        "R2": r2_score(y, oof),
        "cv_r2_mean": np.mean(fold_r2_scores),
        "cv_r2_std": np.std(fold_r2_scores, ddof=1),
        "cv_r2_variance": np.var(fold_r2_scores, ddof=1),
        "RMSE": np.sqrt(mean_squared_error(y, oof)),
        "MAE": mean_absolute_error(y, oof),
        "MAPE": mean_absolute_percentage_error(y, oof),
    })

carbon_metrics_df = pd.DataFrame(carbon_metrics)
carbon_metrics_df = carbon_metrics_df.sort_values(["R2", "cv_r2_variance"], ascending=[False, True]).reset_index(drop=True)
carbon_metrics_df.to_csv(OUT_DIR / "carbon_model_metrics.csv", index=False, encoding="utf-8-sig")
carbon_metrics_df

<!-- CODEx bilingual cell explanation: start -->
### Cell 09 — 碳贡献合并双视图图 / Consolidated dual-view carbon contribution figure

**中文说明**：合并原绝对值和归一化贡献图：左图显示各能源载体总碳排放贡献，右图显示平均 OCEI 贡献及占比，并导出 carbon_breakdown_by_carrier.csv。该 cell 替代原先重复的 Figure 15/16。

**输入与依赖**：读取当前工作流已经生成的数据表或内存对象，并使用统一的绘图样式。

**主要输出**：导出论文或复核用图像文件，并在必要时同步导出支撑图表的数据表。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Consolidates the former absolute and normalised contribution figures: the left panel shows total carbon contribution by carrier, and the right panel shows average OCEI contribution with shares. It exports carbon_breakdown_by_carrier.csv and replaces the previously redundant Figures 15/16.

**Inputs and dependencies**: Uses data tables or in-memory objects already produced in the workflow with a consistent plotting style.

**Main outputs**: Exports manuscript or audit figures and, when needed, the source tables behind the figure.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ---------- 5) Consolidated dual-view carbon contribution figure ----------
carrier_carbon_total = pd.Series({
    "electricity": (df["electricity_kwh_for_carbon"] * EMISSION_FACTORS["electricity"]).sum(),
    "natural_gas": (df["natural_gas_kwh_for_carbon"] * EMISSION_FACTORS["natural_gas"]).sum(),
    "district_heating": (df["district_heating_kwh_for_carbon"] * EMISSION_FACTORS["district_heating"]).sum(),
    "district_cooling": (df["district_cooling_kwh_for_carbon"] * EMISSION_FACTORS["district_cooling"]).sum(),
}, dtype=float).reindex(["electricity", "natural_gas", "district_heating", "district_cooling"]).fillna(0.0)

carrier_labels_zh = {
    "electricity": "电力",
    "natural_gas": "天然气",
    "district_heating": "区域供热",
    "district_cooling": "区域供冷",
}

carrier_ocei_avg = pd.Series({
    carrier: (df[f"{carrier}_kwh_for_carbon"] * EMISSION_FACTORS[carrier] / df["gross_floor_area_m2"]).mean()
    for carrier in carrier_carbon_total.index
}, dtype=float)
carrier_share = carrier_carbon_total / carrier_carbon_total.sum()

carbon_breakdown_df = pd.DataFrame({
    "carrier": carrier_carbon_total.index,
    "carrier_zh": [carrier_labels_zh[c] for c in carrier_carbon_total.index],
    "total_carbon_kgco2e": carrier_carbon_total.values,
    "avg_carbon_kgco2e_per_sample": (carrier_carbon_total / len(df)).values,
    "avg_ocei_kgco2e_m2": carrier_ocei_avg.values,
    "share": carrier_share.values,
})
carbon_breakdown_df.to_csv(OUT_DIR / "carbon_breakdown_by_carrier.csv", index=False, encoding="utf-8-sig")

fig, axes = plt.subplots(1, 2, figsize=(13.2, 5.2), dpi=150)

ax = axes[0]
bars = ax.bar(carbon_breakdown_df["carrier_zh"], carbon_breakdown_df["total_carbon_kgco2e"], color="#4C72B0")
ax.set_title("各能源载体总碳排放贡献")
ax.set_ylabel("年碳排放贡献（kgCO2e/a）")
ax.tick_params(axis="x", rotation=20)
for bar, share in zip(bars, carbon_breakdown_df["share"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f"{share*100:.1f}%", ha="center", va="bottom", fontsize=9)

ax = axes[1]
bars = ax.bar(carbon_breakdown_df["carrier_zh"], carbon_breakdown_df["avg_ocei_kgco2e_m2"], color="#DD8452")
ax.set_title("各能源载体平均 OCEI 贡献")
ax.set_ylabel("OCEI 贡献（kgCO2e/(m2·a)）")
ax.tick_params(axis="x", rotation=20)
for bar, val in zip(bars, carbon_breakdown_df["avg_ocei_kgco2e_m2"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f"{val:.1f}", ha="center", va="bottom", fontsize=9)

fig.tight_layout()
save_figure(fig, FIG_DIR / "carbon_contribution_dual_view.png", dpi=300, bbox_inches="tight")
plt.show()

display(carbon_breakdown_df.round(4))


<!-- CODEx bilingual cell explanation: start -->
### Cell 10 — 碳贡献表展示 / Carbon-contribution table display

**中文说明**：显示按能源载体汇总的总碳排放、样本平均碳排放、平均 OCEI 和占比，供论文表格或补充材料引用。

**输入与依赖**：依赖前序 cell 已经建立的配置、数据对象或函数，请按 notebook 顺序运行。

**主要输出**：生成后续分析所需的中间对象、诊断表、图件或本地输出文件。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Displays total carbon, average carbon per sample, average OCEI, and contribution share by carrier for use in manuscript tables or supplementary materials.

**Inputs and dependencies**: Depends on configuration, data objects, or functions defined by previous cells; run the notebook sequentially.

**Main outputs**: Produces intermediate objects, diagnostic tables, figures, or local output files required downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ---------- 5-Table) Carbon-contribution table display ----------
if "carbon_breakdown_df" not in globals():
    raise RuntimeError("请先运行合并双视图碳贡献图 cell。")

display_cols = [
    "carrier", "carrier_zh", "total_carbon_kgco2e",
    "avg_carbon_kgco2e_per_sample", "avg_ocei_kgco2e_m2", "share"
]
display(carbon_breakdown_df[display_cols].round(4))


<!-- CODEx bilingual cell explanation: start -->
### Cell 11 — 归一化 OCEI 贡献核对 / Normalised OCEI contribution check

**中文说明**：复用合并图中计算的 carrier_ocei_avg，输出归一化 OCEI 贡献表，确保读者无需查看两张重复图也能获得百分比解释。

**输入与依赖**：依赖前序 cell 已经建立的配置、数据对象或函数，请按 notebook 顺序运行。

**主要输出**：生成后续分析所需的中间对象、诊断表、图件或本地输出文件。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Reuses carrier_ocei_avg from the consolidated figure and outputs the normalised OCEI contribution table so readers can obtain percentage interpretation without separate duplicate figures.

**Inputs and dependencies**: Depends on configuration, data objects, or functions defined by previous cells; run the notebook sequentially.

**Main outputs**: Produces intermediate objects, diagnostic tables, figures, or local output files required downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ---------- 5A) Normalised OCEI contribution check ----------
if "carbon_breakdown_df" not in globals():
    raise RuntimeError("Please run the consolidated carbon-contribution cell first.")

normalized_ocei_contribution = carbon_breakdown_df[[
    "carrier", "carrier_zh", "avg_ocei_kgco2e_m2", "share"
]].copy()
normalized_ocei_contribution["share_percent"] = normalized_ocei_contribution["share"] * 100
normalized_ocei_contribution.to_csv(
    OUT_DIR / "normalized_ocei_contribution_by_carrier.csv",
    index=False,
    encoding="utf-8-sig"
)
display(normalized_ocei_contribution.round(4))
print("归一化 OCEI 贡献已并入 carbon_contribution_dual_view.png，不再生成重复的单独图。")


<!-- CODEx bilingual cell explanation: start -->
### Cell 12 — EUI-OCEI 相关性 / EUI-OCEI correlation

**中文说明**：计算 EUI 与 OCEI 的 Pearson 相关，以及单位能耗碳强度与 OCEI 的相关，并绘制 EUI-OCEI 散点图。该分析检验能耗强度和碳强度是否可以相互替代。

**输入与依赖**：依赖前序 cell 已经建立的配置、数据对象或函数，请按 notebook 顺序运行。

**主要输出**：生成后续分析所需的中间对象、诊断表、图件或本地输出文件。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Computes Pearson correlations between EUI and OCEI, and between carbon per kWh and OCEI, then plots the EUI-OCEI scatter. The analysis tests whether energy intensity and carbon intensity can be treated as interchangeable.

**Inputs and dependencies**: Depends on configuration, data objects, or functions defined by previous cells; run the notebook sequentially.

**Main outputs**: Produces intermediate objects, diagnostic tables, figures, or local output files required downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ---------- EUI-OCEI relationship ----------
from scipy.stats import pearsonr

r_eui_ocei, p_eui_ocei = pearsonr(df["eui_kwh_m2"], df["OCEI_kgco2e_m2"])
r_carbon_factor, p_carbon_factor = pearsonr(df["carbon_per_kwh"], df["OCEI_kgco2e_m2"])

fig, ax = plt.subplots(figsize=(6.8, 6.0))
ax.scatter(df["eui_kwh_m2"], df["OCEI_kgco2e_m2"], alpha=0.25)
ax.set_xlabel("EUI（kWh/m2·a）")
ax.set_ylabel("OCEI（kgCO2e/m2·a）")
ax.set_title("EUI 与 OCEI 交叉分析")
fig.tight_layout()
save_figure(fig, FIG_DIR / "eui_ocei_scatter.png", bbox_inches="tight")
plt.show()

print(f"Pearson correlation between EUI and OCEI: r = {r_eui_ocei:.4f}, p = {p_eui_ocei:.4e}")
print(f"Pearson correlation between carbon_per_kwh and OCEI: r = {r_carbon_factor:.4f}, p = {p_carbon_factor:.4e}")

<!-- CODEx bilingual cell explanation: start -->
### Cell 13 — EUI 与 OCEI 排名偏移 / Ranking shift between EUI and OCEI

**中文说明**：比较按 EUI 和 OCEI 排序的样本名次，计算 Top-10% 重叠率、平均绝对名次偏移和最大偏移，并绘制排名散点图。

**输入与依赖**：依赖前序 cell 已经建立的配置、数据对象或函数，请按 notebook 顺序运行。

**主要输出**：生成后续分析所需的中间对象、诊断表、图件或本地输出文件。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Compares sample rankings by EUI and OCEI, computes Top-10% overlap, mean absolute rank shift, and maximum shift, and plots the ranking scatter.

**Inputs and dependencies**: Depends on configuration, data objects, or functions defined by previous cells; run the notebook sequentially.

**Main outputs**: Produces intermediate objects, diagnostic tables, figures, or local output files required downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ---------- 5A) Ranking shift analysis between EUI and OCEI ----------
rank_df = df[["eui_kwh_m2", "OCEI_kgco2e_m2"]].copy()

rank_df["rank_eui"] = rank_df["eui_kwh_m2"].rank(method="first", ascending=True)
rank_df["rank_ocei"] = rank_df["OCEI_kgco2e_m2"].rank(method="first", ascending=True)
rank_df["rank_shift"] = rank_df["rank_ocei"] - rank_df["rank_eui"]
rank_df["abs_rank_shift"] = rank_df["rank_shift"].abs()

top_n = max(1, int(len(rank_df) * 0.10))
top_eui_idx = set(rank_df.nsmallest(top_n, "eui_kwh_m2").index)
top_ocei_idx = set(rank_df.nsmallest(top_n, "OCEI_kgco2e_m2").index)
overlap_ratio = len(top_eui_idx & top_ocei_idx) / top_n

rank_summary = pd.DataFrame([{
    "sample_count": len(rank_df),
    "top_10pct_n": top_n,
    "top10_overlap_ratio": overlap_ratio,
    "mean_abs_rank_shift": rank_df["abs_rank_shift"].mean(),
    "median_abs_rank_shift": rank_df["abs_rank_shift"].median(),
    "max_abs_rank_shift": rank_df["abs_rank_shift"].max(),
}])

rank_summary.to_csv(OUT_DIR / "eui_ocei_rank_shift_summary.csv", index=False, encoding="utf-8-sig")
rank_df.to_csv(OUT_DIR / "eui_ocei_rank_shift_detail.csv", index=False, encoding="utf-8-sig")

display(rank_summary.round(4))

fig, ax = plt.subplots(figsize=(6.6, 6.0))
ax.scatter(rank_df["rank_eui"], rank_df["rank_ocei"], alpha=0.35)
ax.plot([1, len(rank_df)], [1, len(rank_df)], color="red", linewidth=1.2)
ax.set_xlabel("按 EUI 排名")
ax.set_ylabel("按 OCEI 排名")
ax.set_title("EUI 与 OCEI 排名偏移")
fig.tight_layout()
save_figure(fig, FIG_DIR / "eui_ocei_rank_shift.png", bbox_inches="tight")
plt.show()

<!-- CODEx bilingual cell explanation: start -->
### Cell 14 — EUI-OCEI 对比变量集 / Feature set for EUI-OCEI comparison

**中文说明**：定义用于 EUI 与 OCEI SRC 对比的前 20 个候选变量，覆盖生活热水、几何、系统效率、运行时间和围护结构变量。

**输入与依赖**：依赖前序 cell 已经建立的配置、数据对象或函数，请按 notebook 顺序运行。

**主要输出**：生成后续分析所需的中间对象、诊断表、图件或本地输出文件。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Defines the top 20 candidate features for SRC comparison between EUI and OCEI, covering domestic hot water, geometry, system efficiency, operating hours, and envelope variables.

**Inputs and dependencies**: Depends on configuration, data objects, or functions defined by previous cells; run the notebook sequentially.

**Main outputs**: Produces intermediate objects, diagnostic tables, figures, or local output files required downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ---------- 5B-0) Extend to 20 input variables for EUI-OCEI factor comparison ----------
top20 = [
    "dhw_per_person",
    "floor_num",
    "room_count",
    "footprint_area_m2",
    "dhw_temp",
    "boiler_eff",
    "cop_heating",
    "operation_hours",
    "fresh_air_ach",
    "light_power",
    "wwr",
    "cop_cooling",
    "floor_height",
    "aspect_ratio",
    "equip_power",
    "heat_set",
    "room_area",
    "u_wall",
    "shgc_s",
    "orientation_sin",
]

<!-- CODEx bilingual cell explanation: start -->
### Cell 15 — EUI 与 OCEI bootstrap SRC 对比 / Bootstrap SRC comparison for EUI and OCEI

**中文说明**：分别对 EUI 和 OCEI 运行 bootstrap SRC，合并得到变量重要性差值和符号稳定性表，导出 eui_ocei_factor_compare_bootstrap_src.csv。

**输入与依赖**：依赖前序 cell 已经建立的配置、数据对象或函数，请按 notebook 顺序运行。

**主要输出**：生成后续分析所需的中间对象、诊断表、图件或本地输出文件。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Runs bootstrap SRC separately for EUI and OCEI, merges variable-importance differences and sign-stability information, and exports eui_ocei_factor_compare_bootstrap_src.csv.

**Inputs and dependencies**: Depends on configuration, data objects, or functions defined by previous cells; run the notebook sequentially.

**Main outputs**: Produces intermediate objects, diagnostic tables, figures, or local output files required downstream.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ---------- 5B-1) Bootstrap-SRC comparison for EUI and OCEI ----------

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

def run_src_model(df_in, feature_list, target, seed=42, B=BOOTSTRAP_N):
    X = df_in[feature_list].copy().replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median())
    y = df_in[target].copy()

    x_scaler = StandardScaler()
    y_scaler = StandardScaler()

    X_std = x_scaler.fit_transform(X)
    y_std = y_scaler.fit_transform(y.to_numpy().reshape(-1, 1)).ravel()

    model = LinearRegression()
    model.fit(X_std, y_std)
    coef_full = model.coef_

    rng = np.random.default_rng(seed)
    coef_boot = np.zeros((B, X.shape[1]))

    for b in range(B):
        idx = rng.integers(0, len(X), len(X))
        Xb = X.iloc[idx].reset_index(drop=True)
        yb = y.iloc[idx].reset_index(drop=True)

        Xb_std = StandardScaler().fit_transform(Xb)
        yb_std = StandardScaler().fit_transform(yb.to_numpy().reshape(-1, 1)).ravel()

        mb = LinearRegression()
        mb.fit(Xb_std, yb_std)
        coef_boot[b, :] = mb.coef_

    out = pd.DataFrame({
        "feature": feature_list,
        "SRC": coef_full,
        "abs_SRC": np.abs(coef_full),
        "CI_low": np.quantile(coef_boot, 0.025, axis=0),
        "CI_high": np.quantile(coef_boot, 0.975, axis=0),
    })

    out["sign_stable"] = (
        (out["CI_low"] > 0) | (out["CI_high"] < 0)
    )

    return out.sort_values("abs_SRC", ascending=False).reset_index(drop=True)

compare_features = [c for c in top20 if c in df.columns]

src_eui_cmp = run_src_model(df, compare_features, "eui_kwh_m2", seed=42, B=BOOTSTRAP_N)
src_ocei_cmp = run_src_model(df, compare_features, "OCEI_kgco2e_m2", seed=42, B=BOOTSTRAP_N)

factor_compare_df = (
    src_eui_cmp.rename(columns={
        "SRC": "EUI_SRC",
        "abs_SRC": "abs_EUI_SRC",
        "CI_low": "EUI_CI_low",
        "CI_high": "EUI_CI_high",
        "sign_stable": "EUI_sign_stable"
    })
    .merge(
        src_ocei_cmp.rename(columns={
            "SRC": "OCEI_SRC",
            "abs_SRC": "abs_OCEI_SRC",
            "CI_low": "OCEI_CI_low",
            "CI_high": "OCEI_CI_high",
            "sign_stable": "OCEI_sign_stable"
        }),
        on="feature",
        how="inner"
    )
)

factor_compare_df["delta_abs"] = factor_compare_df["abs_OCEI_SRC"] - factor_compare_df["abs_EUI_SRC"]
factor_compare_df["max_abs"] = factor_compare_df[["abs_EUI_SRC", "abs_OCEI_SRC"]].max(axis=1)

factor_compare_df = factor_compare_df.sort_values("max_abs", ascending=False)
factor_compare_df.to_csv(OUT_DIR / "eui_ocei_factor_compare_bootstrap_src.csv", index=False, encoding="utf-8-sig")
display(factor_compare_df.head(20).round(4))

<!-- CODEx bilingual cell explanation: start -->
### Cell 16 — EUI-OCEI 关键因素对比图 / EUI-OCEI key-factor comparison plot

**中文说明**：绘制 EUI 和 OCEI 的绝对 SRC 并列条形图，展示哪些变量在碳指标下权重增强或减弱。

**输入与依赖**：读取当前工作流已经生成的数据表或内存对象，并使用统一的绘图样式。

**主要输出**：导出论文或复核用图像文件，并在必要时同步导出支撑图表的数据表。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Plots side-by-side absolute SRC bars for EUI and OCEI, showing which variables gain or lose importance under the carbon metric.

**Inputs and dependencies**: Uses data tables or in-memory objects already produced in the workflow with a consistent plotting style.

**Main outputs**: Exports manuscript or audit figures and, when needed, the source tables behind the figure.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ---------- 5B) Key-factor comparison between EUI and OCEI ----------
plot_df = factor_compare_df.head(20).iloc[::-1]

fig, ax = plt.subplots(figsize=(8.6, 6.8))
ypos = np.arange(len(plot_df))

ax.barh(ypos - 0.18, plot_df["abs_EUI_SRC"], height=0.35, label="EUI")
ax.barh(ypos + 0.18, plot_df["abs_OCEI_SRC"], height=0.35, label="OCEI")

ax.set_yticks(ypos)
ax.set_yticklabels(plot_df["feature"])
ax.set_xlabel("标准化回归系数绝对值（|SRC|）")
ax.set_title("EUI 与 OCEI 关键因素对比")
ax.legend(frameon=False)

fig.tight_layout()
save_figure(fig, FIG_DIR / "eui_ocei_factor_compare.png", bbox_inches="tight")
plt.show()

<!-- CODEx bilingual cell explanation: start -->
### Cell 17 — OCEI 预测值与仿真值对比 / Predicted-versus-simulated OCEI plots

**中文说明**：对 OCEI 代理模型绘制 out-of-fold 预测值与仿真计算值对比图，标注 R2、CV 方差、RMSE、MAE 和 MAPE。

**输入与依赖**：读取当前工作流已经生成的数据表或内存对象，并使用统一的绘图样式。

**主要输出**：导出论文或复核用图像文件，并在必要时同步导出支撑图表的数据表。

**复现提示**：运行前确认上游输出路径存在；若当前单元生成图件，需同时检查 PNG 预览和 SVG/PDF 矢量文件的文字完整性、标签间距和图例位置。

**English explanation**: Plots out-of-fold predicted versus calculated OCEI for the carbon surrogate models and annotates R2, CV variance, RMSE, MAE, and MAPE.

**Inputs and dependencies**: Uses data tables or in-memory objects already produced in the workflow with a consistent plotting style.

**Main outputs**: Exports manuscript or audit figures and, when needed, the source tables behind the figure.

**Reproducibility note**: Confirm upstream output paths before running. For figure-producing cells, inspect both PNG previews and SVG/PDF vector exports for complete text, label spacing, and legend placement.
<!-- CODEx bilingual cell explanation: end -->


In [ ]:
# ---------- 6) Top two models: predicted versus simulated carbon intensity ----------
for name, pred in carbon_oof.items():
    valid = np.isfinite(pred)
    fig, ax = plt.subplots(figsize=(6.8, 6.2))
    ax.scatter(
        y[valid], pred[valid],
        s=12,
        color="#4C72B0",
        alpha=0.25,
        edgecolors="none",
        rasterized=True
    )
    lo = min(y.min(), np.nanmin(pred))
    hi = max(y.max(), np.nanmax(pred))
    ax.plot([lo, hi], [lo, hi], linewidth=1.2, color='red')
    ax.set_title(f"{name}: 预测碳强度与计算碳强度对比")
    ax.set_xlabel("计算碳强度（kgCO2e/m2·a）")
    ax.set_ylabel("预测碳强度（kgCO2e/m2·a）")
    row = carbon_metrics_df.loc[carbon_metrics_df["model"] == name].iloc[0]

    txt = (
        f"R² = {r2_score(y[valid], pred[valid]):.4f}\n"
        f"CV Var(R²) = {row['cv_r2_variance']:.6f}\n"
        f"RMSE = {np.sqrt(mean_squared_error(y[valid], pred[valid])):.4f}\n"
        f"MAE = {mean_absolute_error(y[valid], pred[valid]):.4f}\n"
        f"MAPE = {mean_absolute_percentage_error(y[valid], pred[valid]):.4f}"
    )
    ax.text(0.03, 0.97, txt, transform=ax.transAxes, va="top", ha="left",
            bbox=dict(boxstyle="round,pad=0.35", facecolor="white", alpha=0.85))
    fig.tight_layout()
    save_figure(fig, FIG_DIR / f"{name}_carbon_pred_vs_sim.png", bbox_inches="tight")
    plt.show()
